# Food Price Watch 04b: forecast test on 2025-26 (retrospective)
Scores "tomorrow = today" (naive), ETS and the LightGBM configuration chosen on 2024 (notebook 04a, fixed in `config/settings.yaml` `forecast_choice`) on weekly origins from 2025-01-01 to 2026-10-24.
2025-26 was already looked at by the earlier run, so every result here is **retrospective**. Every figure sits beside the naive figure.
Also makes the 7-day forecasts from every report day since 2023-06-01 that notebook 05 uses for the alert ranges.
Writes `reports/tables/M3_test_accuracy.csv`, `M4_test_by_item.csv` (errors and ratios only), `processed/forecasts_test.csv` and `processed/forecasts_h7.csv` (contain prices: Drive only). Run top to bottom and paste the last cell's output back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
# stop if Drive is not really mounted (a failed mount leaves an empty /content/drive on the Colab machine)
assert os.path.ismount('/content/drive'), 'Google Drive is not mounted: Runtime > Disconnect and delete runtime, then rerun cell 1'
assert DATA_ROOT.is_dir(), f'{DATA_ROOT} not found in Google Drive (wrong account, or the folder is missing)'
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
# The choice must have been fixed and committed before this notebook scores anything.
import subprocess
choice = SETTINGS.get('forecast_choice') or {}
assert choice.get('config') and choice.get('params'), 'forecast_choice is empty: run 04a and commit its choice first'
choice_commit = subprocess.run(['git', 'log', '--reverse', '--format=%h %ad', '--date=short', '-S', choice['config'], '--', 'config/settings.yaml'],
                               capture_output=True, text=True).stdout.strip()
choice_commit = choice_commit.splitlines()[0] if choice_commit else 'not found in git history'
print('chosen on 2024:', choice['config'], '| choice first committed in', choice_commit)

In [ ]:
import numpy as np
import pandas as pd
from foodwatch import forecast as fc, metrics as mt
D = pd.read_csv(data('data'), parse_dates=['Date'])
h_data = sha256(data('data'))
f = SETTINGS['forecast']; H = f['horizons']
S = fc.prepare(D, H, f['lags'], f['rolling'], f['rolling_min_periods'])
L = fc.stack(S, H)
dates = D.drop_duplicates('report_idx').set_index('report_idx').Date
test_start, test_end = SETTINGS['periods']['test']
origins = fc.period_origins(dates, test_start, test_end, f['origin_step'], max(H))
cols = fc.feature_columns(choice['feature_set'], f['lags'], f['rolling'])
fit_lgbm = fc.make_fit_lgbm(choice['params'], f['rounds'])
Tst = pd.concat([fc.rolling_forecasts(L, origins, cols, fc.fit_naive, 10**9, 'naive'),
                 fc.ets_forecasts(S, L, origins, H),
                 fc.rolling_forecasts(L, origins, cols, fit_lgbm, f['refit_every'], 'lgbm')], ignore_index=True)
print('test origins:', len(origins), '|', dates[origins[0]].date(), 'to', dates[origins[-1]].date())

In [ ]:
# 7-day forecasts from every report day since daily_h7.start, for the alert ranges (notebook 05). Naive = today's price.
dh = f['daily_h7']
F7 = fc.daily_h7(L, dates, cols, fit_lgbm, dh['start'], dh['refit_every'])
F7 = pd.concat([F7, F7.assign(pred=0.0, forecast=lambda x: np.exp(x.lp_origin), model='naive')], ignore_index=True)
F7['origin_date'] = F7.cutoff.map(dates); F7['target_date'] = F7.target_idx.map(dates)

In [ ]:
# Checks: no model fitted after its origin; only 2025-26 targets scored; nothing touches the 2022 gap; naive = 1;
# the 7-day file has one row per item, model and report day with a price (or carried price) since its start.
Ts = mt.scored(Tst)
expected7 = int(S[(S.report_idx.map(dates) >= pd.Timestamp(dh['start'])) & S.lp.notna()].shape[0])
chk = dict(
    test_forecasts=len(Tst), test_scored=len(Ts),
    first_target_date=str(dates[int(Ts.target_idx.min())].date()), last_target_date=str(dates[int(Ts.target_idx.max())].date()),
    target_before_test=int((Ts.target_idx.map(dates) < pd.Timestamp(test_start)).sum()),
    fit_after_origin=int((Tst.fit_cutoff > Tst.cutoff).sum() + (F7.fit_cutoff > F7.cutoff).sum()),
    rows_touching_gap=int(L.loc[L.report_idx.isin(origins), ['touch_gap', 'tgap']].any(axis=1).sum()),
    naive_rel_mae=round(float(mt.gmean_rel(Tst[Tst.model == 'naive']).gmean_rel_mae.max()), 6),
    h7_rows_per_model=int(len(F7) / 2), h7_rows_expected=expected7,
    h7_first_origin=str(F7.origin_date.min().date()), h7_last_origin=str(F7.origin_date.max().date()),
)
chk['ok'] = (chk['target_before_test'] == 0 and chk['fit_after_origin'] == 0 and chk['rows_touching_gap'] == 0
             and chk['naive_rel_mae'] == 1.0 and chk['h7_rows_per_model'] == expected7)
assert chk['ok'], chk

In [ ]:
M3 = (mt.summary(Tst).merge(mt.gmean_rel(Tst), on=['model', 'horizon']).merge(mt.dm_vs_naive(Tst), on=['model', 'horizon'], how='left')
      .assign(period='2025-01-01 to 2026-10-24 (retrospective)')).round(5)
R = mt.rel_mae_by_item(Tst)
acc = mt.scored(Tst).groupby(['model', 'item_id', 'horizon']).apply(
    lambda g: pd.Series(dict(accuracy=1 - float(((g.forecast - g.actual).abs() / g.actual).mean()))), include_groups=False).reset_index()
M4 = R.merge(acc, on=['model', 'item_id', 'horizon']).rename(columns={'mae': 'mae_log', 'naive': 'mae_log_naive', 'rel': 'rel_mae'})
M4 = M4.assign(period='retrospective').round(5)
T = DATA_ROOT / SETTINGS['files']['tables']
write_csv(M3, T / 'M3_test_accuracy.csv')
write_csv(M4, T / 'M4_test_by_item.csv')
h_test = write_csv(Tst, data('forecasts_test'))
h_h7 = write_csv(F7, data('forecasts_h7'))

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
short = lambda s: s.split(' | ')[0]
print('=== 04b forecast test, 2025-01-01 to 2026-10-24: RETROSPECTIVE ===')
print('code commit:', CODE_COMMIT, '| data_8.csv sha256', h_data[:16])
print('model choice (2024 only):', choice['config'], '| choice first committed in', choice_commit)
print('checks:', 'PASS' if chk['ok'] else 'FAIL', {k: v for k, v in chk.items() if k != 'ok'})
print('\nM3 accuracy by model and horizon; rel = MAE / MAE of "tomorrow = today"; dm_stat > 0 = worse than naive; NaN = not computable')
print(M3[['model', 'horizon', 'n', 'mae_log', 'mae_log_naive', 'gmean_rel_mae', 'mae_tk', 'mae_tk_naive', 'accuracy', 'accuracy_naive',
          'direction_acc', 'dm_stat', 'p_holm']].to_string(index=False))
for m in ['ets', 'lgbm']:
    beats = [h for h in H if M3.loc[(M3.model == m) & (M3.horizon == h), 'gmean_rel_mae'].iloc[0] < 1]
    sig = [h for h in H if (M3.loc[(M3.model == m) & (M3.horizon == h), 'dm_stat'].iloc[0] < 0) and (M3.loc[(M3.model == m) & (M3.horizon == h), 'p_holm'].iloc[0] < 0.05)]
    print(f'{m}: below naive at horizons {beats if beats else "none"}; significantly better (DM, Holm p < 0.05) at {sig if sig else "none"}')
print('\nM4 relative MAE by item at horizon 7 (below 1 = better than naive)')
print(M4[M4.horizon == 7].pivot(index='item_id', columns='model', values='rel_mae').rename(index=short).round(3).to_string())
print('\nfiles:', SETTINGS['files']['forecasts_test'], 'sha256', h_test[:16], '|', SETTINGS['files']['forecasts_h7'], 'sha256', h_h7[:16], '| rows', len(F7))